In [ ]:
%load_ext autoreload
%autoreload 2

import torch
import matplotlib.pyplot as plt
import numpy as np
from torch import optim
from scipy.stats import wasserstein_distance_nd

from stochinter.models import PotentialNet, DirectNet
from stochinter.utils import solve_ode

In [ ]:
# parameters
SEED = 44
BATCH_SIZE = 1024
EPOCHS = 2_000
LEARNING_RATE = 3e-3
N_TEST = 1000
STEPS = 30

In [ ]:
torch.manual_seed(SEED)
np.random.seed(SEED)

batch_size = BATCH_SIZE
epochs = EPOCHS

net_potential = PotentialNet()
net_direct = DirectNet()

opt_pot = optim.Adam(net_potential.parameters(), lr=LEARNING_RATE)
opt_dir = optim.Adam(net_direct.parameters(), lr=LEARNING_RATE)

print("Training of models")

source_mean = torch.tensor([-5.0, 0.0], dtype=torch.float32)
target_mean = torch.tensor([5.0, 0.0], dtype=torch.float32)

for epoch in range(epochs):
    x0 = torch.randn(batch_size, 2) + source_mean
    x1 = torch.randn(batch_size, 2) + target_mean
    
    t = torch.rand(batch_size, 1) * 0.98 + 0.01
    z = torch.randn(batch_size, 2)

    
    x_t = (1 - t) * x0 + t * x1  + torch.sqrt(2*t*(1-t))*z 
    
    mean_t = (1.0 - t) * source_mean + t * target_mean
    var_t = (1.0 - t)**2 + t**2 + 2*t*(1-t)
    score = -(x_t - mean_t) / var_t
    
    omega = 5
    v_tornado = omega * torch.stack([-score[:, 1], score[:, 0]], dim=-1)
    
    v_target = (x1 - x0 + z*(1-2*t) / torch.sqrt(2*t*(1-t))) + v_tornado 

    opt_pot.zero_grad()
    v_pred_pot = net_potential(x_t, t)
    loss_pot = torch.mean((v_pred_pot - v_target)**2)
    loss_pot.backward()
    opt_pot.step()
    
    opt_dir.zero_grad()
    v_pred_dir = net_direct(x_t, t)
    loss_dir = torch.mean((v_pred_dir - v_target)**2)
    loss_dir.backward()
    opt_dir.step()
    
    if (epoch+1) % 500 == 0:
        print(f"Epoch {epoch+1:4d} | Scalar loss: {loss_pot.item():.4f} | Direct loss: {loss_dir.item():.4f}")

In [ ]:
@torch.no_grad
def velocity_pot(x, t):
    return net_potential(x,t).detach()

@torch.no_grad()
def velocity_dir(x, t):
    return net_direct(x, t)

In [ ]:
print("Running inference...")

n_test = N_TEST
x_start = torch.randn(n_test, 2) + source_mean
steps = STEPS

x_gen_pot, pot_trajectories = solve_ode(x_start, velocity_pot, steps=steps, return_trajectories=True)
x_gen_dir, dir_trajectories = solve_ode(x_start, velocity_dir, steps=steps, return_trajectories=True)

print("Inference done!")

In [ ]:
target_mean = torch.tensor([5.0, 0.0], dtype=torch.float32)
source_mean = torch.tensor([-5.0, 0.0], dtype=torch.float32)

x_target = torch.randn(batch_size, 2) + target_mean
x_source = torch.randn(batch_size, 2) + source_mean

x_source_np = x_source.cpu().numpy()
x_target_np = x_target.cpu().numpy()

x_pot_np = x_gen_pot.cpu().detach().numpy()
x_dir_np = x_gen_dir.cpu().detach().numpy()

def plot_scatter(ax, data, title):
    ax.scatter(data[:, 0], data[:, 1], s=4, alpha=0.3, color='black')
    ax.set_title(title, fontsize=14)
    ax.set_xlim(-10, 10)
    ax.set_ylim(-10, 10)
    ax.set_aspect('equal')
    ax.grid(True, linestyle=':', alpha=0.5)

In [ ]:
fig1, axes1 = plt.subplots(1, 2, figsize=(10, 5))

plot_scatter(axes1[0], x_source_np, "Initial Data ($X_0$)")
plot_scatter(axes1[1], x_target_np, "Target ($X_1$)")

fig1.tight_layout()
plt.savefig('experiment_2_true_distr', dpi=600)
plt.show()

In [ ]:
x_start_np = x_start.numpy()

In [ ]:
fig2, axes2 = plt.subplots(1, 2, figsize=(10, 5))
n_to_plot = 10 

plot_scatter(axes2[0], np.concatenate([x_dir_np, x_start_np]), "Generated (DirectNet)")
for i in range(n_to_plot):
    axes2[0].plot(dir_trajectories[:, i, 0], dir_trajectories[:, i, 1], color='gray', alpha=0.3, linewidth=1)

plot_scatter(axes2[1], np.concatenate([x_pot_np, x_start_np]), "Generated (PotentialNet)")
for i in range(n_to_plot):
    axes2[1].plot(pot_trajectories[:, i, 0], pot_trajectories[:, i, 1], color='gray', alpha=0.3, linewidth=1)


fig2.tight_layout()
fig2.savefig('experiment2.png', dpi=600)
plt.show()

In [ ]:
print("Wasserstein distance for potential net:", wasserstein_distance_nd(x_target_np, x_pot_np))
print("Wasserstein distance for direct net:", wasserstein_distance_nd(x_target_np, x_dir_np))